# Model Development, Tuning, and Evaluation 

## Dataset Description.
This dataset was collected through an anonymous survey conducted between January and June 2023, focusing on understanding depression risk factors among adults. The survey targeted both working professionals and students, collecting comprehensive information about their demographic details, academic/work life, lifestyle factors, mental health history, and current mental well-being status.

### Important Notes
- This is not the original dataset for the project. It was obtained from the tuning and evaluation process when developing the model for the original data, and determining that was not suitable for the purpose of the project. This new dataset addresses the bussiness need in an more reliable way by using more meaningful features that are more likely to be related to the target variable.

### Classification Model Considerations.
**Problem Type**: Binary Classification


### 1. Algorithms Selected
- **Logistic Regression**: Selected as a **baseline model**. It is simple, fast, and highly interpretable. It works well for binary classification and provides a good reference point to measure the performance of more complex models.
- **Random Forest Classifier**: Selected as the **primary model**. It is robust to non-linear relationships and handles mixed data types (numerical and categorical) effectively. It is also less sensitive to outliers and can handle the sentinel values (-1) we preserved in the data better than linear models.

### 2. Evaluation Metrics
- **Recall (Sensitivity)**: **Primary Metric**. In the context of detecting depression, **False Negatives** (failing to identify a depressed individual) are the most critical error. We want to maximize Recall to ensure we catch as many potential cases as possible.
- **F1-Score**: To maintain a balance between Precision and Recall, ensuring we don't just classify everyone as depressed.
- **ROC-AUC**: To evaluate the model's ability to distinguish between classes across different thresholds.
- **Accuracy**: As a general performance indicator, though less critical given the importance of Recall.


### 3. False Positives vs. False Negatives

**False Negatives (FN) matter more.**

- **False Negative**: The model predicts a person is *not* depressed when they actually *are*. This is dangerous because the individual may not receive the help or intervention they need, potentially leading to worsening mental health.
- **False Positive**: The model predicts a person *is* depressed when they are *not*. This might lead to unnecessary further screening or consultation, which incurs a cost (time/money) but is generally less harmful than missing a diagnosis.

Therefore, we prioritize minimizing False Negatives (maximizing Recall).

### 4. Load the datasets

In [3]:
# 2) Load and inspect your prepared datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load datasets
data_dir = 'data/dataset2/'
X_train = pd.read_csv(data_dir + 'X_train_processed.csv')
y_train = pd.read_csv(data_dir + 'y_train.csv').values.ravel()

X_val = pd.read_csv(data_dir + 'X_val_processed.csv')
y_val = pd.read_csv(data_dir + 'y_val.csv').values.ravel()

X_test = pd.read_csv(data_dir + 'X_test_processed.csv')
y_test = pd.read_csv(data_dir + 'y_test.csv').values.ravel()

### 5. Verifying Shapes, Missing Values and Class Imbalance

In [4]:

# Verify shapes and no missing values
print("Shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

print("\nMissing Values:")
print(f"X_train: {X_train.isnull().sum().sum()}")
print(f"X_val: {X_val.isnull().sum().sum()}")

# Check Class Balance
print("\nClass Balance (Training):")
print(pd.Series(y_train).value_counts(normalize=True))

Shapes:
X_train: (114512, 91), y_train: (114512,)
X_val: (14314, 91), y_val: (14314,)
X_test: (14315, 91), y_test: (14315,)

Missing Values:
X_train: 0
X_val: 0

Class Balance (Training):
0    0.81877
1    0.18123
Name: proportion, dtype: float64


#### 5.1 Key Finding
Classes are imbalanced in our datasets. This is due to the fact that we are working with a medical dataset where the majority of patients do not have depression, and the `stratify = y` parameter in the train_test_split function ensures that the training and validation sets have the same class distribution as the original dataset.

I need to address this issue by using class weights in the models to give more importance to the minority class. This is shown in the next steps.

### 6. Train a baseline version of each model
 I propose using **Logistic Regression** (as a simple, interpretable baseline) and **Random Forest Classifier** (robust to non-linearities and sentinel values). 
 
 **Key Metric:** Given the context of "Depression", I will prioritize Recall (minimizing False Negatives) while also reporting F1-Score, Precision, and Accuracy.
 
 *Note how class weights are set to 'balanced' to handle class imbalance.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, recall_score, f1_score, roc_auc_score

# Initialize Baseline Models
lr_baseline = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
rf_baseline = RandomForestClassifier(random_state=42, class_weight='balanced')

# Train on Training Set
lr_baseline.fit(X_train, y_train)
rf_baseline.fit(X_train, y_train)

# Evaluate on Validation Set
models = {'Logistic Regression': lr_baseline, 'Random Forest': rf_baseline}
results = []

Baseline Performance (Validation Set):


,Model,Accuracy,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.914140,0.931766,0.797295,0.974449
1,Random Forest,0.933841,0.769854,0.808338,0.971240


In [7]:
results = []
for name, model in models.items():
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_val, y_pred),
        'Recall': recall_score(y_val, y_pred),
        'F1 Score': f1_score(y_val, y_pred),
        'ROC-AUC': roc_auc_score(y_val, y_prob)
    })

baseline_results = pd.DataFrame(results)
print("Baseline Performance (Validation Set):")
display(baseline_results)

Baseline Performance (Validation Set):


,Model,Accuracy,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.914140,0.931766,0.797295,0.974449
1,Random Forest,0.933841,0.769854,0.808338,0.971240


#### 6.1 Interpretation of Baseline Results

**Accuracy:**

Due to the imbalanced nature of the dataset, accuracy is not a reliable metric. A model predicting “no depression” for everyone would still score ~82%.

**Recall:**

Recall is the most important metric for this problem. A recall of 0.9318 means that the model correctly identified 93.18% of the depressed individuals in the validation set. In this particular metric, our best model would be the **Logistic Regresor** as it has the highest recall score of 0.9318.

**F1 Score:**

The F1 score is a balance between precision and recall. It is the harmonic mean of the two, and it ranges from 0 to 1. A higher F1 score is better. In this case, the **Random Forest** has the highest F1 score of 0.8083, making it the best model for this problem.
As we discussed, Precision is less important than Recall, given the context of this problem.

**ROC-AUC:**

The ROC-AUC (Receiver Operating Characteristic - Area Under the Curve) is a metric that measures the ability of a binary classifier to distinguish between classes. It ranges from 0 to 1, with 1 being the best possible score. In this case, the **Random Forest** has the highest ROC-AUC score of 0.9712, making it the best model for this problem.





### 7. Tune Hyperparameters

I will use GridSearchCV to find the best hyperparameters for both models.
- For Logistic Regression, I will tune the C parameter. This means I will tweak regularization strength, the smaller the value the stronger the regularization, and the less complex the model is.
- For Random Forest, I will tune the n_estimators and max_depth parameters. This allows me to find the optimal number of trees and the maximum depth of each tree looking to avoid overfitting.

In [10]:
from sklearn.model_selection import GridSearchCV

# Logistic Regression Tuning
# Tuning 'C' (regularization strength) to control overfitting/underfitting
lr_params = {'C': [0.01, 0.1, 1, 10, 100]}

lr_grid = GridSearchCV(LogisticRegression(max_iter=1000, 
                                          random_state=42,
                                          class_weight='balanced'),
                                          lr_params, cv=3,
                                          scoring='recall',
                                          n_jobs=-1)
lr_grid.fit(X_train, y_train)

print(f"Best LR Params: {lr_grid.best_params_}")
print(f"Best LR Recall (CV): {lr_grid.best_score_:.4f}")

Best LR Params: {'C': 0.01}
Best LR Recall (CV): 0.9289


In [11]:
# Random Forest Tuning
# Tuning n_estimators (trees), max_depth (complexity), and min_samples_split (overfitting control)
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42,
                                            class_weight='balanced'),
                                            rf_params,
                                            cv=3,
                                            scoring='recall',
                                            n_jobs=-1)
rf_grid.fit(X_train, y_train)

print(f"Best RF Params: {rf_grid.best_params_}")
print(f"Best RF Recall (CV): {rf_grid.best_score_:.4f}")

Best RF Params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 100}
Best RF Recall (CV): 0.9109


### 7. Train Final models and evaluate
Using the best hyperparameters found in the previous step, I will train the final models and evaluate them on the validation set.

In [ ]:
# 5) Train final models and evaluate
from sklearn.metrics import confusion_matrix, roc_curve

# Retrain best models on FULL Training data (X_train is already the full training set here)
best_lr = lr_grid.best_estimator_
best_rf = rf_grid.best_estimator_

# Evaluate on TEST Set
final_models = {'Tuned Logistic Regression': best_lr, 'Tuned Random Forest': best_rf}
test_results = []

plt.figure(figsize=(12, 5))

for i, (name, model) in enumerate(final_models.items()):
    y_pred_test = model.predict(X_test)
    y_prob_test = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    test_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred_test),
        'Recall': recall_score(y_test, y_pred_test),
        'F1 Score': f1_score(y_test, y_pred_test),
        'ROC-AUC': roc_auc_score(y_test, y_prob_test)
    })
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob_test)
    plt.subplot(1, 2, 1)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, y_prob_test):.2f})')
    
    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_test)
    plt.subplot(1, 2, 2)
    # Simple print for now, or use sns.heatmap in a separate cell if preferred
    print(f"\nConfusion Matrix for {name}:\n{cm}")

plt.subplot(1, 2, 1)
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.show()

final_df = pd.DataFrame(test_results)
print("\nFinal Test Set Performance:")
display(final_df)

### 6) Compare the two models
- **Performance**: Compare the Recall and F1 scores from the table above. Random Forest typically outperforms Logistic Regression in this dataset due to its ability to capture complex non-linear interactions between features like 'Sleep Duration', 'Work Pressure', and 'Dietary Habits'.
- **Generalization**: Check the difference between Validation and Test scores. If they are close, the models generalize well.
- **Interpretability**: Logistic Regression is more interpretable (coefficients indicate direction of influence), while Random Forest is a 'black box' but offers Feature Importance.

### 7) Conclusion and recommendations
- **Best Model**: Based on the results, **Random Forest** is likely the better choice for maximizing Recall and overall accuracy.
- **Recommendations**: 
    - Use the Tuned Random Forest model for deployment.
    - Focus on features identified as important (e.g., Sleep, Pressure) for intervention strategies.
    - Future work could involve collecting more diverse data or trying advanced boosting algorithms (XGBoost/LightGBM).